# Build Results Fact

1. Read silver `results` table
1. Read silver `sprints` table
1. Add new column `session_type` with values `RACE` or `SPRINT`
1. UNION `results` and `sprints`
1. Derive additional columns
    - is_win -> Indicates that the driver own the race
    - is_podium -> Indicates that the driver scored a podium result (1, 2, 3)
    - has_points -> Indicates that the driver has scored points
1. Write the transformed data to gold `fact_session_results` table



#### Entity Relationship Diagram - Formula1 Silver Schema

![Formula1 Silver Data.png](../z-course-images/formula1-silver-data-erd.png "Formula1 Silver Data.png")


#### Entity Relationship Diagram - Formula1 Gold Schema

![Formula1 Gold Data.png](../z-course-images/formula1-gold-data-erd.png "Formula1 Gold Data.png")

In [0]:
%run ../00-common/01.environment-config

In [0]:
target_table = f"{catalog_name}.{gold_schema}.fact_session_results"

In [0]:
from pyspark.sql import functions as F

#### Step 1 - Read source tables
- `silver.results`
- `silver.sprints`

In [0]:
results_df = (
    spark.table(f"{catalog_name}.{silver_schema}.results")
         .withColumn("session_type", F.lit("RACE"))
         .drop("race_name", "race_date", "ingestion_timestamp", "source_file")
)

In [0]:
sprints_df = (
    spark.table(f"{catalog_name}.{silver_schema}.sprints")
         .withColumn("session_type", F.lit("SPRINT"))
         .drop("race_name", "race_date", "ingestion_timestamp", "source_file")
)

#### Step 2 - UNION `results` and `sprints`

In [0]:
results_sprints_df = results_df.unionByName(sprints_df)

#### Step 3 - Add dervied columns
1. is_win -> Indicates that the driver own the race
1. is_podium -> Indicates that the driver scored a podium result (1, 2, 3)
1. has_points -> Indicates that the driver has scored points


In [0]:
fact_session_results_df = (
    results_sprints_df
        .withColumn("is_win", F.col("final_position") == 1)
        .withColumn("is_podium", F.col("final_position").between(1, 3))
        .withColumn("has_points", F.col("points") > 0)
)

In [0]:
display(fact_session_results_df.filter("season = 2025"))

#### Step 4 - Write the transformed data to the `gold` `fact_session_results` table

In [0]:
(
    fact_session_results_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
)

In [0]:
display(spark.table(target_table))